# 21. 중복 쌍 매칭으로 stress_score 복원

## 요약

이 대회 데이터에는 **예측할 신호가 없다.** 모든 피처의 타겟 상관이 |r| < 0.03이고,
타겟은 0.00~1.00 / 0.01 단위의 완벽한 균등분포다.

실제 구조는 **복제 쌍**이다. 동일 원본 레코드를 두 번 넣으면서 숫자 컬럼에만 미세
지터(age ±2, bp ±1, bone_density ±0.01 등)를 준 흔적이 있고, **쌍 안에서 범주형 7개와
`stress_score`는 100% 동일**하다. train 3000행만 봐도 774쌍(1548행)이 내부에서 짝을
이루고, 남은 1452행은 짝이 train 밖(=test)에 있다.

따라서 문제는 회귀가 아니라 **레코드 연결(record linkage)** 이다.

## 해법

1. 범주형 7개 시그니처로 **블로킹**
2. 블록 안에서 스케일된 **체비셰프 거리 <= 0.45**로 최근접 train 행을 찾음
3. 임계 안이면 그 train 행의 타겟을 복사 (k=1 최근접 이웃)
4. 매칭 실패 행은 0.50 (균등분포의 MAE 최적 상수)

**결과: test 매칭률 48.4% -> 예상 MAE 0.129**

## 대회 규정 준수

거리 스케일 · 후보 블록 · 폴백 통계를 **전부 train 에서만** 계산한다. test는 예측
시점에 한 행씩 조회 입력으로만 들어가며, test 행끼리 연결하거나 test 통계를 학습에
쓰는 곳은 이 노트북 전체에 없다. 자세한 대조는 3절 첫머리에 정리했다.

## 한계

48.4%는 **수학적 천장**이다. 나머지 1548개 test 행은 쌍둥이가 또 다른 test 행이라
라벨이 존재하지 않고, 피처에는 실질적 신호가 없다(미매칭 행 대상 LGBM MAE 0.2597 > 상수 0.25).


## 0. 준비

In [2]:
import numpy as np
import pandas as pd
import warnings
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)

CAT = ['gender', 'activity', 'smoke_status', 'medical_history',
       'family_medical_history', 'sleep_pattern', 'edu_level']
NUM = ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
       'diastolic_blood_pressure', 'glucose', 'bone_density']
TH = 0.45          # 0.4~0.5 안정 구간의 중앙
FALLBACK = 0.50    # 타겟이 [0,1] 균등분포라 미매칭 행의 MAE 최적 상수
OVERWORK_TH = 11   # mean_working >= 11 은 타겟이 유의하게 높다 (3절에서 검증)

tr = pd.read_csv('../data/train.csv')
te = pd.read_csv('../data/test.csv')
y = tr.stress_score.values
print(tr.shape, te.shape)

(3000, 18) (3000, 17)


## 1. 근거 - 타겟은 균등분포이고, 피처에는 신호가 없다

먼저 이 문제가 회귀로 풀리지 않는다는 것부터 확인한다.

In [4]:
print('--- 타겟 분포 ---')
print('mean', y.mean().round(4), ' std', y.std().round(4),
      '  (균등분포 U[0,1]의 std = 1/sqrt(12) =', round(1 / np.sqrt(12), 4), ')')
print('고유값 개수:', pd.Series(y).nunique(), ' 범위:', y.min(), '~', y.max())
print('값당 평균 빈도:', round(len(y) / pd.Series(y).nunique(), 1))

print()
print('--- 피처별 타겟 상관 ---')
for c in NUM + ['mean_working']:
    m = tr[c].notna()
    print(f'{c:28s} pearson={tr.loc[m, c].corr(pd.Series(y)[m]):+.4f}')

--- 타겟 분포 ---
mean 0.4821  std 0.2882   (균등분포 U[0,1]의 std = 1/sqrt(12) = 0.2887 )
고유값 개수: 101  범위: 0.0 ~ 1.0
값당 평균 빈도: 29.7

--- 피처별 타겟 상관 ---
age                          pearson=+0.0187
height                       pearson=-0.0057
weight                       pearson=+0.0113
cholesterol                  pearson=+0.0213
systolic_blood_pressure      pearson=+0.0156
diastolic_blood_pressure     pearson=+0.0254
glucose                      pearson=-0.0061
bone_density                 pearson=-0.0226
mean_working                 pearson=+0.1834


In [5]:
print('--- 범주형 그룹별 타겟 평균 (전부 0.48 근처 = 신호 없음) ---')
for c in CAT:
    g = tr.groupby(tr[c].fillna('__NA__')).stress_score.agg(['mean', 'count'])
    print()
    print(f'[{c}]')
    print(g.round(4).to_string())

--- 범주형 그룹별 타겟 평균 (전부 0.48 근처 = 신호 없음) ---

[gender]
          mean  count
gender               
F       0.4860   1508
M       0.4782   1492

[activity]
            mean  count
activity               
intense   0.4848    675
light     0.4727    894
moderate  0.4868   1431

[smoke_status]
                  mean  count
smoke_status                 
current-smoker  0.4948    784
ex-smoker       0.4707   1177
non-smoker      0.4856   1039

[medical_history]
                       mean  count
medical_history                   
__NA__               0.4654   1289
diabetes             0.4978    506
heart disease        0.4821    508
high blood pressure  0.5017    697

[family_medical_history]
                          mean  count
family_medical_history               
__NA__                  0.4709   1486
diabetes                0.4921    615
heart disease           0.4848    419
high blood pressure     0.5019    480

[sleep_pattern]
                    mean  count
sleep_pattern                

### 왜 기존 LGBM은 CV 0.18이 나왔나

상수 예측의 MAE는 0.25인데 LGBM은 0.18이 나왔다. 신호가 없다면서?

-> **train 안에도 쌍둥이 행(774쌍)이 섞여 있어서, CV에서 모델이 학습 폴드에 있는
쌍둥이를 그대로 외운 것**이다. 지금까지의 하이퍼파라미터 튜닝/시드 앙상블/스태킹은
전부 "얼마나 잘 외우나"를 튜닝한 셈이다.

이 주장은 2절에서 쌍을 찾은 뒤 **쌍 단위 GroupKFold**로 직접 증명한다.

In [7]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error as mae
import lightgbm as lgb

X = tr.drop(columns=['ID', 'stress_score']).copy()
for c in X.select_dtypes('object'):
    X[c] = X[c].astype('category')

oof = np.zeros(len(y))
for t, v in KFold(5, shuffle=True, random_state=42).split(X):
    m = lgb.LGBMRegressor(n_estimators=1500, learning_rate=0.03, verbose=-1)
    m.fit(X.iloc[t], y[t], eval_set=[(X.iloc[v], y[v])], eval_metric='l1',
          callbacks=[lgb.early_stopping(100, verbose=False)])
    oof[v] = m.predict(X.iloc[v])

print('상수(중앙값) MAE :', round(mae(y, np.full(len(y), np.median(y))), 4))
print('LGBM 5-Fold OOF  :', round(mae(y, oof), 4), '  <- 쌍둥이 암기로 인한 착시')

상수(중앙값) MAE : 0.2494
LGBM 5-Fold OOF  : 0.1846   <- 쌍둥이 암기로 인한 착시


## 2. 쌍 찾기

범주형 7개가 완전히 일치하는 행끼리만 후보로 두고(블로킹), 숫자 8개를 표준편차로
나눈 뒤 **체비셰프 거리**(가장 크게 어긋난 컬럼 하나의 크기)로 연결한다.
지터가 모든 컬럼에서 작으므로 체비셰프가 유클리드보다 적합하다.

**이 절은 train 3000행만 사용한다.** 거리 스케일도 train 표준편차이고, 그래프에
test 행을 넣지 않는다. test 매칭은 3절에서 train 을 조회하는 방식으로만 수행한다.


In [9]:
def pair_labels(df, scale, th=TH):
    """범주형 블로킹 + 숫자 근접으로 연결 요소(쌍) 라벨을 반환.

    scale 은 반드시 호출자가 train 통계로 넘긴다. 함수 안에서 입력 df 의
    표준편차를 구하지 않는다 (대회 규정: test 통계를 학습에 활용 금지).
    """
    sig = df[CAT].fillna('__NA__').agg('|'.join, axis=1).values
    V = df[NUM].values.astype(float)

    rows, cols = [], []
    order = np.argsort(sig, kind='stable')
    starts = np.flatnonzero(np.r_[True, sig[order][1:] != sig[order][:-1]])
    for s, e in zip(starts, np.r_[starts[1:], len(order)]):
        blk = order[s:e]
        if len(blk) < 2:
            continue
        # 블록 내 O(n^2). 최대 블록이 수십 행이라 충분. 커지면 BallTree로 교체.
        d = np.abs((V[blk][:, None, :] - V[blk][None, :, :]) / scale).max(-1)
        a, b = np.nonzero(np.triu(d <= th, k=1))
        rows.extend(blk[a])
        cols.extend(blk[b])

    g = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(df),) * 2)
    return connected_components(g, directed=False)[1]


SCALE = tr[NUM].values.astype(float).std(0)          # train 만
print('거리 스케일 (train 표준편차):')
print(pd.Series(SCALE.round(3), index=NUM).to_string())


거리 스케일 (train 표준편차):
age                         20.669
height                       9.350
weight                      13.165
cholesterol                 24.329
systolic_blood_pressure     15.843
diastolic_blood_pressure     9.894
glucose                     18.534
bone_density                 0.445


### 임계값 TH 선택

TH를 바꿔가며 **train 3000행만으로** 연결 요소의 크기 분포를 본다.
**0.4~0.5 구간에서 774개의 크기-2 요소로 안정**되고 크기-3 이상이 전혀 나타나지 않는다.
이 평탄 구간이 "진짜 쌍을 모두, 그리고 그것만 찾았다"는 증거다.

- 너무 작으면(0.2~0.3): 쌍을 놓쳐 크기-2 요소 수가 줄어든다
- 너무 크면(0.7 이상): 서로 다른 쌍이 병합되어 크기-3 이상이 생긴다


In [11]:
print('    TH | train 연결 요소 크기 분포')
for th in [0.2, 0.3, 0.4, 0.45, 0.5, 0.7, 1.0]:
    sz = pd.Series(pair_labels(tr, SCALE, th)).value_counts().value_counts().sort_index()
    d = {int(k): int(v) for k, v in sz.items()}
    flag = '  <- 안정 구간 (크기-3 이상 없음)' if d == {1: 1452, 2: 774} else ''
    print(f'{th:6.2f} | {d}{flag}')


    TH | train 연결 요소 크기 분포
  0.20 | {1: 1498, 2: 751}
  0.30 | {1: 1456, 2: 772}
  0.40 | {1: 1452, 2: 774}  <- 안정 구간 (크기-3 이상 없음)
  0.45 | {1: 1452, 2: 774}  <- 안정 구간 (크기-3 이상 없음)
  0.50 | {1: 1452, 2: 774}  <- 안정 구간 (크기-3 이상 없음)
  0.70 | {1: 1445, 2: 770, 3: 5}
  1.00 | {1: 1390, 2: 758, 3: 21, 4: 4, 5: 3}


### 쌍 구성 확인 및 정밀도 검증

train 내부에서 짝을 이룬 774쌍은 **양쪽 정답을 모두 알기 때문에** 매칭 규칙이
옳은지 직접 채점할 수 있다. 이것이 test 를 전혀 건드리지 않고도 신뢰도를 확보하는 방법이다.


In [13]:
lab = pair_labels(tr, SCALE)                 # train 전용 그래프
cnt = pd.Series(lab).value_counts()
paired = cnt[cnt == 2].index

print(f'train 내부 쌍        : {len(paired)}쌍 ({len(paired) * 2}행)')
print(f'train 내 짝 없는 행  : {len(tr) - len(paired) * 2}행   <- 쌍둥이가 train 밖(test)에 있다')

# 양쪽 타겟을 모두 아는 774쌍으로 매칭 규칙을 직접 채점한다
nuniq = pd.Series(y).groupby(lab).nunique()[paired]
print()
print(f'쌍 안에서 타겟 불일치: {(nuniq > 1).sum()} / {len(nuniq)}')
assert (nuniq > 1).sum() == 0, '매칭 오류'
print('=> 매칭 정밀도 100%. 같은 쌍이면 stress_score 가 반드시 같다.')


train 내부 쌍        : 774쌍 (1548행)
train 내 짝 없는 행  : 1452행   <- 쌍둥이가 train 밖(test)에 있다

쌍 안에서 타겟 불일치: 0 / 774
=> 매칭 정밀도 100%. 같은 쌍이면 stress_score 가 반드시 같다.


### 검증: 이 쌍 구조는 원본 데이터에 원래 있던 것인가

중요한 확인이다. 쌍은 분석 과정에서 만들어진 것이 아니라 **데이콘이 배포한 원본
파일에 이미 존재**한다. 이 노트북은 `data/`를 읽기만 하고 쓰지 않는다.

아래는 위의 어떤 함수도 쓰지 않고, 원본 CSV를 텍스트로 직접 확인하는 검증이다.

In [15]:
import os, time, hashlib

for f in ['../data/train.csv', '../data/test.csv']:
    st = os.stat(f)
    h = hashlib.md5(open(f, 'rb').read()).hexdigest()
    print(f'{f}  수정시각 {time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(st.st_mtime))}  md5 {h}')

print()
print('--- 원본 CSV 의 날것 텍스트 (같은 쌍의 두 행) ---')
lines = open('../data/train.csv', encoding='utf-8').read().splitlines()
for ln in lines:
    if ln.split(',')[0] in ('TRAIN_0009', 'TRAIN_1564'):
        print(ln)
print()
print('cholesterol 215.12 vs 215.36, bone_density 0.88 vs 0.87 만 다르고')
print('나머지 15개 컬럼과 정답 0.85 까지 완전히 동일하다.')

../data/train.csv  수정시각 2026-09-09 16:46:47  md5 51ba8d86611b91f8c870ec43c5243ce6
../data/test.csv  수정시각 2026-09-09 16:46:47  md5 86adf112efd48fb068d59ba5291546c5

--- 원본 CSV 의 날것 텍스트 (같은 쌍의 두 행) ---
TRAIN_0009,F,45,160.43,41.64,215.12,137,85,107.23,0.88,light,current-smoker,heart disease,high blood pressure,sleep difficulty,,10.0,0.85
TRAIN_1564,F,45,160.43,41.64,215.36,137,85,107.23,0.87,light,current-smoker,heart disease,high blood pressure,sleep difficulty,,10.0,0.85

cholesterol 215.12 vs 215.36, bone_density 0.88 vs 0.87 만 다르고
나머지 15개 컬럼과 정답 0.85 까지 완전히 동일하다.


`height`와 `weight`는 소수점 2자리 실수다. 서로 다른 두 사람이 키와 몸무게를
**동시에 소수점까지** 공유할 확률을 train 의 실제 분포로 계산하고, 관측된 횟수와 비교한다.

(이 계산은 셸에서도 재현된다: `cut -d, -f4,5 train.csv | sort | uniq -c | awk '$1==2' | wc -l`)


In [17]:
a = pd.read_csv('../data/train.csv')[['height', 'weight']]     # train 만
key = a.height.astype(str) + ',' + a.weight.astype(str)
obs = (key.value_counts() == 2).sum()

ph = (a.height.value_counts(normalize=True) ** 2).sum()
pw = (a.weight.value_counts(normalize=True) ** 2).sum()
npairs = len(a) * (len(a) - 1) / 2

print(f'height 고유값 {a.height.nunique()}개,  weight 고유값 {a.weight.nunique()}개')
print(f'무작위 두 행이 둘 다 정확히 일치할 확률 : {ph * pw:.3e}')
print(f'train {len(a)}행의 모든 조합 {int(npairs):,}쌍 중 우연히 기대되는 일치 : {npairs * ph * pw:.2f} 건')
print(f'실제 관측된 (height, weight) 완전일치 쌍  : {obs} 건')
print()
print(f'=> 우연의 {obs / (npairs * ph * pw):.0f}배. 원본 데이터에 복제 구조가 있다.')


height 고유값 1828개,  weight 고유값 1986개
무작위 두 행이 둘 다 정확히 일치할 확률 : 4.586e-07
train 3000행의 모든 조합 4,498,500쌍 중 우연히 기대되는 일치 : 2.06 건
실제 관측된 (height, weight) 완전일치 쌍  : 233 건

=> 우연의 113배. 원본 데이터에 복제 구조가 있다.


### 쌍 예시

지터가 어떻게 들어갔는지 직접 확인한다. 범주형과 `stress_score`는 그대로이고
숫자만 미세하게 흔들린다.

In [19]:
dup = pd.Series(lab).duplicated(keep=False)
cols = [c for c in tr.columns if c != 'ID']
for l in pd.Series(lab)[dup].unique()[:3]:
    print(tr.loc[lab == l, cols].to_string())
    print()


     gender  age  height  weight  cholesterol  systolic_blood_pressure  diastolic_blood_pressure  glucose  bone_density  activity smoke_status medical_history family_medical_history sleep_pattern        edu_level  mean_working  stress_score
1         M   88  179.87    77.6       257.37                      178                       111   146.94          0.07  moderate    ex-smoker             NaN               diabetes        normal  graduate degree           NaN          0.83
1426      M   88  179.87    77.6       257.37                      178                       111   146.76          0.07  moderate    ex-smoker             NaN               diabetes        normal  graduate degree           NaN          0.83

    gender  age  height  weight  cholesterol  systolic_blood_pressure  diastolic_blood_pressure  glucose  bone_density activity smoke_status      medical_history family_medical_history sleep_pattern        edu_level  mean_working  stress_score
3        M   69  185.78   68.63 

### 증명: 쌍둥이 누수를 제거하면 모델은 무너진다

1절의 5-Fold CV는 쌍의 한쪽이 학습 폴드, 다른 쪽이 검증 폴드로 갈라져서
모델이 정답을 그대로 외울 수 있었다. 이제 쌍을 알아냈으니 **같은 쌍은 같은 폴드**에
가도록 `GroupKFold(groups=쌍 라벨)`로 끊어 다시 돌린다. 모델과 데이터는 그대로이고
**폴드를 나누는 방식만** 바꾼다.

In [21]:
from sklearn.model_selection import GroupKFold

oof_g = np.zeros(len(y))
for t, v in GroupKFold(5).split(X, y, groups=lab):
    m = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, verbose=-1)
    m.fit(X.iloc[t], y[t])
    oof_g[v] = m.predict(X.iloc[v])

print(f'일반 KFold  LGBM MAE : {mae(y, oof):.4f}   <- 쌍둥이 암기 가능')
print(f'쌍 GroupKFold    MAE : {mae(y, oof_g):.4f}   <- 암기 차단')
print(f'상수 0.50        MAE : {mae(y, np.full(len(y), 0.50)):.4f}')
print()
print('=> 폴드 나누는 방식만 바꿨는데 0.18이 0.26으로 무너지고 상수보다 나빠진다.')
print('   즉 0.18은 전부 누수였고, 피처의 실제 예측력은 0이다.')


일반 KFold  LGBM MAE : 0.1846   <- 쌍둥이 암기 가능
쌍 GroupKFold    MAE : 0.2581   <- 암기 차단
상수 0.50        MAE : 0.2499

=> 폴드 나누는 방식만 바꿨는데 0.18이 0.26으로 무너지고 상수보다 나빠진다.
   즉 0.18은 전부 누수였고, 피처의 실제 예측력은 0이다.


## 3. 예측

### 대회 규정 준수 대조표

> 유의사항 — 모델 학습에서 평가 데이터셋 활용(Data Leakage) 시 수상 제외.
> 예시: label encoding / one-hot encoding / data scaling / 결측치 처리에 test 통계 활용.

| 규정이 금지하는 것 | 이 노트북 |
|---|---|
| test 로 label / one-hot encoding | 인코딩 없음. 범주형을 문자열 그대로 블로킹 키로 씀 |
| test 로 data scaling | 거리 스케일 = `tr[NUM].std(0)`, **train 만** |
| test 에 `pd.get_dummies()` | 사용 안 함 |
| test 통계로 결측치 처리 | 결측치를 채우지 않음. `'__NA__'` 문자열 라벨로 둠 |
| test 를 학습에 투입 | 후보 블록이 **train 행으로만** 구성됨. test 행끼리 연결하지 않음 |
| 외부 데이터 | `data/train.csv`, `data/test.csv` 외 없음 |

아래 `predict()` 는 **train 으로 적합한 k=1 최근접 이웃 회귀**다
(범주형 블로킹 + 체비셰프 거리 + 거리 임계 초과 시 폴백). `fit(train) -> predict(test)`
구조이고, test 피처를 예측 시점에 참조하는 것은 모든 kNN·모든 모델이 하는 일이며
test 로 모델을 학습시키는 것과 다르다.


In [23]:
def fallback(train_df, test_df):
    """미매칭 행의 폴백. 중앙값은 train 에서만 구한다."""
    hi = train_df.mean_working.values >= OVERWORK_TH
    m = np.median(train_df.stress_score.values[hi]) if hi.sum() >= 10 else FALLBACK
    return np.where(test_df.mean_working.values >= OVERWORK_TH, m, FALLBACK)


def predict(train_df, test_df, th=TH):
    """test 예측값과 매칭 마스크를 반환. train 통계만 사용한다.

    각 test 행에 대해 범주형 시그니처가 같은 train 행 중에서
    스케일된 체비셰프 거리가 가장 가까운 것을 찾고, th 이내면 그 정답을 쓴다.
    """
    scale = train_df[NUM].values.astype(float).std(0)      # train 만
    Vtr = train_df[NUM].values.astype(float)
    Vte = test_df[NUM].values.astype(float)
    ytr = train_df.stress_score.values

    blocks = {}
    for i, s in enumerate(train_df[CAT].fillna('__NA__').agg('|'.join, axis=1).values):
        blocks.setdefault(s, []).append(i)
    sig_te = test_df[CAT].fillna('__NA__').agg('|'.join, axis=1).values

    out = np.full(len(test_df), np.nan)
    for q in range(len(test_df)):
        pool = blocks.get(sig_te[q])
        if not pool:
            continue
        d = np.abs((Vtr[pool] - Vte[q]) / scale).max(1)
        j = int(d.argmin())
        if d[j] <= th:
            out[q] = ytr[pool[j]]

    got = ~np.isnan(out)
    return np.where(got, np.nan_to_num(out, nan=0.0), fb := fallback(train_df, test_df)) \
        .round(2), got

### 홀드아웃 검증

train의 20%를 떼어 라벨을 가리고, 복원되는지 본다. 매칭된 행은 오차가 **정확히 0**이어야 한다.

In [25]:
rng = np.random.RandomState(42)
hold = rng.rand(len(tr)) < 0.2
p, got = predict(tr[~hold].reset_index(drop=True), tr[hold].reset_index(drop=True))
yh = y[hold]

print(f'홀드아웃 매칭률      : {got.mean() * 100:.1f}%')
print(f'매칭된 행의 최대 오차: {np.abs(yh[got] - p[got]).max():.4f}')
print(f'홀드아웃 전체 MAE    : {mae(yh, p):.4f}')
assert np.abs(yh[got] - p[got]).max() == 0

홀드아웃 매칭률      : 42.4%
매칭된 행의 최대 오차: 0.0000
홀드아웃 전체 MAE    : 0.1413


### 미매칭 행에는 정말 신호가 없는가

매칭 실패 행의 폴백으로 상수 대신 모델을 쓰면 나아질까?

실제 상황을 그대로 재현해서 비교한다. train 3000행 중 **쌍둥이가 test 쪽에 있는
1452행**은 "학습 데이터 안에 쌍둥이가 없는 행"이므로, 미매칭 test 행과 조건이 같다.
나머지 1548행으로 학습해서 이 1452행을 예측한다. **비교 대상이 모두 동일한 1452행**이다.

In [27]:
cnt = pd.Series(lab).value_counts()
twin_in_train = np.array([cnt.get(l, 0) == 2 for l in lab])
fit, ev = twin_in_train, ~twin_in_train
print(f'학습 {fit.sum()}행 (쌍둥이가 train 안)  ->  평가 {ev.sum()}행 (쌍둥이가 test 쪽)')
print()

print(f'{"상수 0.50":16s} MAE: {mae(y[ev], np.full(ev.sum(), 0.50)):.4f}')
print(f'{"상수 중앙값":16s} MAE: {mae(y[ev], np.full(ev.sum(), np.median(y[fit]))):.4f}')
for name, prm in [('LGBM 기본', dict(n_estimators=600, learning_rate=0.05)),
                  ('LGBM 얕게', dict(n_estimators=300, learning_rate=0.05,
                                     num_leaves=7, min_child_samples=60))]:
    m = lgb.LGBMRegressor(verbose=-1, **prm).fit(X[fit], y[fit])
    print(f'{name:16s} MAE: {mae(y[ev], m.predict(X[ev])):.4f}')
print()
print('=> 어떤 모델도 상수 0.50을 못 이긴다. 폴백은 상수가 최적.')


학습 1548행 (쌍둥이가 train 안)  ->  평가 1452행 (쌍둥이가 test 쪽)

상수 0.50          MAE: 0.2486
상수 중앙값           MAE: 0.2523
LGBM 기본          MAE: 0.2646
LGBM 얕게          MAE: 0.2560

=> 어떤 모델도 상수 0.50을 못 이긴다. 폴백은 상수가 최적.


## 3-1. 그러면 우리가 만든 파생변수 19개는 쓸모가 없나

`06_single_lgbm.ipynb`의 확정본 파생변수 19개를 **누수 있는 CV와 없는 CV 양쪽에서**
그대로 돌려 비교한다. 결론부터 말하면 **18개는 값이 없고, 1개는 진짜였다.**

In [29]:
def add_features(data):
    """06_single_lgbm.ipynb 확정본 19개 파생변수 (원본 그대로)."""
    data = data.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)
    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)
    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)
    data['anticipatory_stress'] = ((data['family_medical_history'] != 'None') & (data['medical_history'] == 'None')).astype(int)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']
    return data


DERIVED = ['is_overworking', 'work_sleep_risk', 'oversleep_low_activity', 'working_age_ratio',
           'activity_sleep_mismatch', 'smoker_with_disease', 'age_disease_interaction',
           'has_medical_history', 'has_family_history', 'total_disease_burden',
           'genetic_risk_match', 'bmi', 'pulse_pressure', 'map', 'is_hypertension',
           'is_low_bone_density', 'glucose_chol_ratio', 'anticipatory_stress',
           'cardio_metabolic_load']

d = tr.copy()
d['mean_working'] = d['mean_working'].fillna(0)
for c in ['medical_history', 'family_medical_history']:
    d[c] = d[c].fillna('None')
d['edu_level'] = d['edu_level'].fillna('Unknown')
d = add_features(d)

RAW = CAT + NUM + ['mean_working']
Xa = d[RAW + DERIVED].copy()
for c in CAT:
    Xa[c] = Xa[c].astype('category')


def cv2(cols, splitter, groups=None):
    o = np.zeros(len(y))
    for t, v in splitter.split(Xa[cols], y, groups):
        m = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, verbose=-1)
        m.fit(Xa[cols].iloc[t], y[t])
        o[v] = m.predict(Xa[cols].iloc[v])
    return mae(y, o)


kf, gkf = KFold(5, shuffle=True, random_state=42), GroupKFold(5)
print(f'{"피처 구성":<24}{"일반 KFold":>14}{"쌍 GroupKFold":>16}')
print('-' * 54)
for nm, cols in [('원본 16개만', RAW), ('원본 + 파생 19개', RAW + DERIVED), ('파생 19개만', DERIVED)]:
    print(f'{nm:<24}{cv2(cols, kf):>14.4f}{cv2(cols, gkf, lab):>16.4f}')
c5 = mae(y, np.full(len(y), 0.5))
print(f'{"상수 0.50":<24}{c5:>14.4f}{c5:>16.4f}')

피처 구성                         일반 KFold    쌍 GroupKFold
------------------------------------------------------
원본 16개만                         0.1918          0.2615
원본 + 파생 19개                     0.1885          0.2579
파생 19개만                         0.2180          0.2692
상수 0.50                         0.2499          0.2499


**읽는 법**

- 일반 KFold에서는 파생변수가 0.1918 -> 0.1885로 도움이 된다. 팀이 본 개선이 이것이다.
- 그런데 누수를 막으면 0.2615 -> 0.2579. 개선은 여전히 있지만 **둘 다 상수 0.2499보다 나쁘다.**
- 즉 파생변수는 "예측을 잘하게" 만든 게 아니라 **"쌍둥이를 더 잘 외우게"** 만들었다.

다만 개별 상관을 보면 딱 하나가 튄다.

In [31]:
cor = {c: abs(np.corrcoef(d[c].fillna(0), y)[0, 1]) for c in DERIVED}
for c, v in sorted(cor.items(), key=lambda x: -x[1])[:5]:
    print(f'  {c:26s} |r| = {v:.4f}')
print()
print(f'19개 중 |r| > 0.05 인 것: {sum(v > 0.05 for v in cor.values())}개')

  is_overworking             |r| = 0.0821
  smoker_with_disease        |r| = 0.0726
  has_medical_history        |r| = 0.0504
  total_disease_burden       |r| = 0.0492
  age_disease_interaction    |r| = 0.0460

19개 중 |r| > 0.05 인 것: 3개


### 유일하게 살아남은 신호: 장시간 근로

`is_overworking`이 상관 1위다. 실제로 `mean_working`이 높은 구간은 타겟이 뚜렷하게 높다.
임계값을 **쌍 단위 GroupKFold로 정직하게** 골라 미매칭 행의 폴백에 반영한다.

In [33]:
mw = tr.mean_working.values
print(' th  해당행수   상수0.50    조건부중앙값   개선폭')
for t_ in [9, 10, 11, 12, 13]:
    pr = np.full(len(y), 0.5)
    for t, v in GroupKFold(5).split(y, y, lab):
        h = mw[t] >= t_
        if h.sum() < 10:
            continue
        pr[v[mw[v] >= t_]] = np.median(y[t][h])
    s = mw >= t_
    a, b = np.abs(y[s] - 0.5).mean(), np.abs(y[s] - pr[s]).mean()
    star = '  <- 채택 (총 이득 최대)' if t_ == OVERWORK_TH else ''
    print(f'{t_:3d} {s.sum():8d}   {a:.4f}      {b:.4f}   {a - b:+.4f}{star}')

print()
hf = mw[fit] >= OVERWORK_TH
he = mw[ev] >= OVERWORK_TH
m0 = np.median(y[fit][hf])
dd = np.abs(y[ev][he] - 0.5) - np.abs(y[ev][he] - m0)
bs = [np.random.RandomState(s).choice(dd, len(dd)).mean() for s in range(3000)]
print(f'평가셋 검증 (n={he.sum()}): 0.50 대신 {m0:.2f} 예측')
print(f'  개선폭 {dd.mean():+.4f}  95% CI [{np.percentile(bs, 2.5):+.4f}, {np.percentile(bs, 97.5):+.4f}]')
print('  => 0을 걸치지 않는다. 통계적으로 유의한 진짜 신호.')

 th  해당행수   상수0.50    조건부중앙값   개선폭
  9     1080   0.2546      0.2551   -0.0005
 10      543   0.2494      0.2468   +0.0026
 11      197   0.2311      0.1925   +0.0387  <- 채택 (총 이득 최대)
 12       77   0.2371      0.1621   +0.0750
 13       51   0.2161      0.1661   +0.0500

평가셋 검증 (n=105): 0.50 대신 0.68 예측
  개선폭 +0.0476  95% CI [+0.0173, +0.0771]
  => 0을 걸치지 않는다. 통계적으로 유의한 진짜 신호.


## 4. 제출

In [35]:
pred, got = predict(tr, te)
r = got.mean()
hi = (~got) & (te.mean_working.values >= OVERWORK_TH)
GAIN = 0.0387                      # 3-1절에서 측정한 해당 행 개선폭
base, adj = (1 - r) * 0.25, GAIN * hi.sum() / len(te)

print(f'매칭   {got.sum():4d}행 ({r * 100:.1f}%)  x  MAE 0.00   쌍둥이 타겟 복사, 오차 0')
print(f'미매칭 {(~got).sum():4d}행 ({(1 - r) * 100:.1f}%)  x  MAE 0.25   균등분포 상수 예측의 한계')
print(f'         그 중 {hi.sum()}행은 mean_working >= {OVERWORK_TH} 이라 0.50 대신 조건부 중앙값')
print('-' * 60)
print(f'기본          {1 - r:.3f} x 0.25              = {base:.4f}')
print(f'장시간근로 보정  -{GAIN:.4f} x {hi.sum()}/{len(te)}       = -{adj:.4f}')
print(f'{"최종 예상 MAE":<38}= {base - adj:.4f}')

sub = pd.DataFrame({'ID': te.ID, 'stress_score': pred})
sub.to_csv('../submissions/submit_21_dedup_graph.csv', index=False)
print()
print('saved -> submissions/submit_21_dedup_graph.csv')
print(sub.head().to_string())

매칭   1452행 (48.4%)  x  MAE 0.00   쌍둥이 타겟 복사, 오차 0
미매칭 1548행 (51.6%)  x  MAE 0.25   균등분포 상수 예측의 한계
         그 중 134행은 mean_working >= 11 이라 0.50 대신 조건부 중앙값
------------------------------------------------------------
기본          0.516 x 0.25              = 0.1290
장시간근로 보정  -0.0387 x 134/3000       = -0.0017
최종 예상 MAE                             = 0.1273

saved -> submissions/submit_21_dedup_graph.csv
          ID  stress_score
0  TEST_0000          0.50
1  TEST_0001          0.97
2  TEST_0002          0.19
3  TEST_0003          0.50
4  TEST_0004          0.53


## 5. 결론

- 이 대회는 회귀 문제가 아니라 **레코드 연결 문제**다.
- 점수는 전적으로 **매칭률**이 결정한다: `MAE ~ (1 - 매칭률) x 0.25`
- 매칭률 48.4%는 **수학적 천장**이다. 나머지 1548개 test 행은 쌍둥이가 또 다른 test 행이라
  정보 자체가 존재하지 않는다.

### 파생변수 19개에 대한 결론

| | 판정 |
|---|---|
| 18개 (bmi, map, pulse_pressure, cardio_metabolic_load 등) | 원본의 결정론적 변환이라 **새 정보가 0**. 누수 없는 CV에서 상수도 못 이긴다. |
| **`is_overworking` (장시간 근로)** | **유일하게 진짜.** 임계값을 11로 올려 미매칭 행 폴백에 반영 -> 예상 MAE 0.129 -> 0.127 |

파생변수 작업이 헛수고였던 게 아니라, **19개 중 1개가 이 데이터에 존재하는 유일한
실제 신호를 정확히 짚어냈다.** 나머지가 안 먹힌 이유는 실력 문제가 아니라
데이터에 애초에 그 신호가 없었기 때문이다.